# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [26]:
planeData = pd.read_csv('data/AviationData.csv', encoding = 'cp1252')
planeData.info()

C:\Users\Jason.DESKTOP-MNVBQQF\AppData\Local\Temp\ipykernel_3052\244227971.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  planeData = pd.read_csv('data/AviationData.csv', encoding = 'cp1252')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [27]:
recentPlanes = planeData[
    pd.to_datetime(planeData['Event.Date'], format = '%Y-%m-%d', errors = 'coerce') 
    > '1983-01-01']
recentProfessionalPlanes = recentPlanes[recentPlanes['Amateur.Built'] == 'No']
# smallPlanes = recentProfessionalPlanes[recentProfessionalPlanes[]]
large = recentProfessionalPlanes[recentProfessionalPlanes['Number.of.Engines'] >= 2] 
#No perfect way to tell the size of plane, so I am basing size by Engine Count(before I looked below)
small = recentProfessionalPlanes[recentProfessionalPlanes['Number.of.Engines'] < 2]

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.


In [28]:
cols = [
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]
#I Assume that the total number of passengers will be the total of all of the injured + non injured
recentProfessionalPlanes['Number.of.Passengers'] = (
    recentProfessionalPlanes[cols].sum(axis=1)
)

C:\Users\Jason.DESKTOP-MNVBQQF\AppData\Local\Temp\ipykernel_3052\3592069261.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recentProfessionalPlanes['Number.of.Passengers'] = (


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
recentProfessionalPlanes = recentProfessionalPlanes.dropna(
    subset=['Aircraft.damage']
) #used .dropna with a subset to drop whole row from dataframe

recentProfessionalPlanes['Destroyed'] = (recentProfessionalPlanes['Aircraft.damage'] == "Destroyed")
#checks if aircraft.damage is destroyed, if it is, that expression will return true in the new column

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [ ]:
#1. first i am going to drop any nan values
#2. second, I will investigate some of the values by using .iloc and seeing some example values
#3 group the makes by their size
#4 Filter out makes less than the threshold requirement

recentProfessionalPlanes = recentProfessionalPlanes.dropna(subset = ['Make'])
# for i in range(1,5):
#     print(recentProfessionalPlanes['Make'].iloc[i])
grouped = recentProfessionalPlanes.groupby('Make').size()
grouped = grouped[grouped >= 50] #current threshold, used to keep makes with a reasonable number
print(grouped)

Make
AERO COMMANDER      69
AERONCA            149
AIR TRACTOR         91
AIR TRACTOR INC    217
AIRBUS             105
                  ... 
TAYLORCRAFT         61
Taylorcraft        303
Waco               113
Weatherly           78
Wsk Pzl Mielec      77
Length: 103, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [55]:
recentProfessionalPlanes = recentProfessionalPlanes.dropna(subset = ['Model'])
# for i in range(1,30):
#     print(f'{i}. {recentProfessionalPlanes['Make'].iloc[i]}')
#     print(f'{i}. {recentProfessionalPlanes['Model'].iloc[i]}')

# dupes = recentProfessionalPlanes.groupby('Model')['Make'].nunique()
# dupes = dupes[dupes> 1]
# print(dupes)
x = recentProfessionalPlanes.groupby('Model')['Make'].nunique()
recentProfessionalPlanes['UniqueID'] = (recentProfessionalPlanes['Make'] + ' ' + recentProfessionalPlanes['Model'])

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [66]:
cols = [
'Engine.Type',
'Weather.Condition',
'Purpose.of.flight',
'Broad.phase.of.flight'
]
#Standardize capitalization and spacing
for col in cols:
    recentProfessionalPlanes[col] = ( recentProfessionalPlanes[col].str.strip().str.title() )
#standardize number of engines as integer(used int64 so nans are not included)   
recentProfessionalPlanes['Number.of.Engines'] = ( recentProfessionalPlanes['Number.of.Engines'].astype('Int64'))
#found that Air Race Show and Air Race/Show are both used
recentProfessionalPlanes['Purpose.of.flight'] = ( 
                            recentProfessionalPlanes['Purpose.of.flight'].replace({
                            "Air Race/Show": "Air Race Show"
                            }))
recentProfessionalPlanes['Engine.Type'] = (
    recentProfessionalPlanes['Engine.Type']
    .replace('UNK', 'Unknown')
)

x = recentProfessionalPlanes.dropna(subset = ['Broad.phase.of.flight'])
x = x.groupby('Broad.phase.of.flight').size() 
print(x)
# #checking 
# for i in range(1,20):
#     print(x['Purpose.of.flight'].iloc[i])


Broad.phase.of.flight
Approach        5383
Climb           1628
Cruise          8475
Descent         1515
Go-Around       1212
Landing        13390
Maneuvering     6735
Other             86
Standing         706
Takeoff        10201
Taxi            1655
Unknown          417
dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [71]:
recentProfessionalPlanes.isna().sum()
recentProfessionalPlanes.isna().mean() * 100
recentProfessionalPlanes = recentProfessionalPlanes.dropna(
    axis=1,
    thresh=len(recentProfessionalPlanes) * 0.5
)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [72]:
recentProfessionalPlanes.to_csv(
    'cleaned_plane_data.csv',
    index=False
)